In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/ezzeddinesayed52@gmail.com/FMCG-DE_Project/Scripts/Setups/utilities

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","orders","Data Source")

data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

base_path = (
    f"r2://sportsbar@"
    f"eefc59d684bff864c9de01ed688857a0"
    f".r2.cloudflarestorage.com/"
    f"{data_source}"
)
landing_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"

print(base_path)
print(landing_path)
print(processed_path)

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"


In [0]:
df = spark.read\
    .options(header=True, inferSchema=True)\
    .csv(f"{landing_path}/*.csv")\
    .withColumn("read_timestamp",F.current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")

print("Total Rows:", df.count())
display(df.limit(5))

In [0]:
display(df.limit(20))

In [0]:
df.write\
    .mode("append")\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .saveAsTable(bronze_table)

In [0]:
#moving the files from landing to processed
for file in dbutils.fs.ls(landing_path):
  dbutils.fs.mv(file.path,f"{processed_path}/{file.name}",True)

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
display(df_orders.limit(20))

In [0]:
date_col = F.col("order_placement_date")
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(date_col, "yyyy/MM/dd"),
        F.try_to_date(date_col, "dd-MM-yyyy"),
        F.try_to_date(date_col, "dd/MM/yyyy"),
        F.try_to_date(date_col, "yyyy-MM-dd"),
        F.try_to_date(
            F.regexp_replace(date_col, r"^[A-Za-z]+,\s*", ""),
            "MMMM dd, yyyy"
        )
    )
)

display(df_orders)

In [0]:
df_orders.select(
    [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_orders.columns
    ]
).show()

In [0]:

df_orders = df_orders.filter(F.col("order_qty").isNotNull())
df_orders = df_orders.withColumn(
    "customer_id",
    F.coalesce(
        F.expr("try_cast(customer_id as bigint)"),
        F.lit(999999)
    ).cast("string")
)
display(df_orders)

In [0]:
df_orders.select(
    [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_orders.columns
    ]
).show()

In [0]:
df_orders.printSchema()

In [0]:
df_orders = df_orders.withColumn(
    "product_id", F.col("product_id").cast("string")
)

df_orders.printSchema()

In [0]:
df_orders.groupBy(df_orders.columns).count().filter(F.col("count") >1).count()

In [0]:
df_orders = df_orders.dropDuplicates(df_orders.columns)
df_orders.groupBy(df_orders.columns).count().filter(F.col("count") >1).count()

In [0]:
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
 display(df_orders.limit(20))

In [0]:
df_orders.groupBy("order_id","customer_id","product_id","order_placement_date","order_qty").count().filter(
    F.col("count") > 1
).show()

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_orders.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias("silver").merge(df_orders.alias("bronze"), "silver.order_id = bronze.order_id AND silver.customer_id = bronze.customer_id AND silver.product_id = bronze.product_id and silver.order_placement_date = bronze.order_placement_date AND silver.order_qty = bronze.order_qty")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll().execute()

In [0]:
df_silver = spark.sql(f"SELECT * FROM {silver_table}")    
df_silver.printSchema()

In [0]:
df_gold = df_silver.select(
    F.col("order_id"),
    F.col("order_placement_date").alias("date"),
    F.col("product_id").alias("product_code"),
    F.col("customer_id").alias("customer_code"),
    F.col("order_qty").alias("sold_quantity")

)
df_gold.show()

In [0]:
df_gold = df_gold.dropDuplicates(["order_id", "customer_code", "product_code", "date"])

if not (spark.catalog.tableExists(gold_table)):
    df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark,gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.order_id = gold.order_id AND source.customer_code = gold.customer_code AND source.product_code = gold.product_code and source.date = gold.date AND source.sold_quantity = gold.sold_quantity")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll().execute()

Merge with Parent table

In [0]:
df_child = spark.sql(f"SELECT * FROM {gold_table}")
df_child.printSchema()

In [0]:
df_child = df_child.select("date","product_code","customer_code","sold_quantity")
df_child.show()


In [0]:
df_child = df_child.withColumn(
    "date",
    F.trunc(F.col("date"), "month")
)
df_child = df_child.groupBy("date","product_code","customer_code").agg(F.sum("sold_quantity").alias("sold_quantity"))
display(df_child)

In [0]:
gold_parent_delta = DeltaTable.forName(spark,f"{catalog}.{gold_schema}.fact_{data_source}")
gold_parent_delta.alias("target").merge(df_child.alias("source"),"source.date = target.date AND source.product_code = target.product_code AND source.customer_code = target.customer_code")\
.whenMatchedUpdateAll()\
.whenNotMatchedInsertAll().execute()